## Example 1: google_resumable_media functionality test 1

In [ ]:
from google.resumable_media import requests, common
import io

# Example 1: Basic resumable upload functionality
print("Example 1: Basic resumable upload functionality")

try:
    # Explore package structure
    print(f"Google resumable media package loaded successfully")
    
    # Check available modules
    import google.resumable_media
    attrs = [attr for attr in dir(google.resumable_media) if not attr.startswith('_')]
    print(f"Available modules: {attrs}")
    assert {'common', 'requests', 'UPLOAD_CHUNK_SIZE'} <= set(attrs), attrs
    
    # Upload URL and basic configuration
    upload_url = "https://www.googleapis.com/upload/storage/v1/b/bucket/o"
    print(f"Example upload URL: {upload_url}")
    
    # Create a simple data stream for demonstration
    data = b"Hello, this is test data for resumable upload"
    stream = io.BytesIO(data)
    print(f"Test data size: {len(data)} bytes")
    
    # Common upload parameters
    media_type = "text/plain"
    total_bytes = len(data)
    
    print(f"Media type: {media_type}")
    print(f"Total bytes: {total_bytes}")
    
    # Resumable upload configuration
    chunk_size = common.UPLOAD_CHUNK_SIZE
    print(f"Default chunk size: {chunk_size} bytes")
    # a resumable upload's chunk size must be a multiple of 256 KiB (measured: 1000 raises ValueError); a new upload
    # object starts unfinished with nothing sent
    assert chunk_size > 0 and chunk_size % (256 * 1024) == 0, chunk_size
    upload = requests.ResumableUpload(upload_url, chunk_size)
    assert (upload.upload_url, upload.chunk_size, upload.finished, upload.bytes_uploaded) == (upload_url, chunk_size, False, 0)
    
    # Upload states
    upload_states = [
        "UPLOAD_STATE_UNSPECIFIED",
        "NOT_STARTED", 
        "RESUMABLE",
        "FINAL"
    ]
    
    print(f"Upload states:")
    for state in upload_states:
        print(f"  • {state}")
    
    print("Example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"google_resumable_media example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: google_resumable_media functionality test 2

In [ ]:
from google.resumable_media.requests import ResumableUpload
import io
import requests

# Example 2: ResumableUpload class demonstration
print("Example 2: ResumableUpload class demonstration")

try:
    # Create resumable upload instance
    upload_url = "https://www.googleapis.com/upload/storage/v1/b/mybucket/o"
    chunk_size = 256 * 1024  # 256KB chunks
    
    print(f"Setting up resumable upload:")
    print(f"  Upload URL: {upload_url}")
    print(f"  Chunk size: {chunk_size} bytes")
    
    # Create upload instance (demonstration only)
    upload = ResumableUpload(upload_url, chunk_size)
    print(f"ResumableUpload instance created")
    
    # Upload properties
    print(f"Upload state: {upload.finished}")
    print(f"Invalid: {upload.invalid}")
    print(f"Bytes uploaded: {upload.bytes_uploaded}")
    print(f"Total bytes: {upload.total_bytes}")
    
    # Test data for upload
    test_data = b"This is test data for resumable upload demonstration"
    stream = io.BytesIO(test_data)
    
    print(f"\nTest data prepared:")
    print(f"  Size: {len(test_data)} bytes")
    print(f"  Content: {test_data[:20]}...")
    
    # Upload methods (demo without actual HTTP calls)
    methods = [
        "initiate() - Start the resumable upload",
        "consume_next_chunk() - Upload next chunk",
        "resume() - Resume interrupted upload",
        "recover() - Recover upload state"
    ]
    
    print(f"\nAvailable upload methods:")
    for method in methods:
        print(f"  • {method}")
    
    # Chunk size configurations
    chunk_sizes = [
        256 * 1024,    # 256KB
        1024 * 1024,   # 1MB  
        8 * 1024 * 1024,  # 8MB
        16 * 1024 * 1024  # 16MB
    ]
    
    print(f"\nRecommended chunk sizes:")
    for size in chunk_sizes:
        print(f"  • {size:,} bytes ({size // 1024}KB)")
    
    print("Example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"google_resumable_media example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: google_resumable_media functionality test 3

In [ ]:
from google.resumable_media.requests import Download, ChunkedDownload
import io

# Example 3: Download and ChunkedDownload functionality
print("Example 3: Download and ChunkedDownload functionality")

try:
    # Simple download setup
    media_url = "https://storage.googleapis.com/bucket/object"
    print(f"Setting up download from: {media_url}")
    
    # Create a download instance. Headers are passed at construction; the object keeps them
    # privately, so the example reports the headers it set.
    custom_headers = {"x-goog-request-reason": "demo-example"}
    download = Download(media_url, headers=custom_headers)
    print(f"Download instance created")
    print(f"Download finished: {download.finished}")
    assert (download.media_url, download.finished) == (media_url, False), download
    
    # Download properties
    print(f"\nDownload properties:")
    print(f"  URL: {download.media_url}")
    print(f"  Headers: {len(custom_headers)} custom headers")
    
    # Chunked download for large files: nothing is fetched until consume_next_chunk() runs
    chunk_size = 1024 * 1024  # 1MB chunks
    stream = io.BytesIO()
    chunked_download = ChunkedDownload(media_url, chunk_size, stream)
    
    print(f"\nChunked download setup:")
    print(f"  Chunk size: {chunked_download.chunk_size:,} bytes")
    print(f"  Finished: {chunked_download.finished}")
    print(f"  Bytes downloaded: {chunked_download.bytes_downloaded}")
    print(f"  Total bytes: {chunked_download.total_bytes}")
    assert (chunked_download.chunk_size, chunked_download.finished, chunked_download.bytes_downloaded) == (chunk_size, False, 0)
    assert chunked_download.total_bytes is None, chunked_download.total_bytes
    
    # Range request headers
    range_examples = [
        "bytes=0-1023",      # First 1KB
        "bytes=1024-2047",   # Second 1KB  
        "bytes=-1024",       # Last 1KB
        "bytes=1024-"        # From 1KB to end
    ]
    
    print(f"\nRange request examples:")
    for range_header in range_examples:
        print(f"  Range: {range_header}")
    
    # Download methods
    download_methods = [
        "consume() - Download entire content",
        "consume_next_chunk() - Download next chunk"
    ]
    
    print(f"\nDownload methods:")
    for method in download_methods:
        print(f"  • {method}")
    assert callable(download.consume) and callable(chunked_download.consume_next_chunk)
    
    # Common HTTP status codes for resumable media
    status_codes = {
        200: "OK - Complete download",
        206: "Partial Content - Chunk downloaded",
        416: "Range Not Satisfiable",
        404: "Not Found"
    }
    
    print(f"\nRelevant HTTP status codes:")
    for code, description in status_codes.items():
        print(f"  {code}: {description}")
    
    print("Example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"google_resumable_media example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: google_resumable_media functionality test 4

In [ ]:
from google.resumable_media import _helpers, common
import hashlib
import io

# Example 4: Upload helpers and checksum validation
print("Example 4: Upload helpers and checksum validation")

try:
    # Content range parsing
    print("Setting up content range and checksum examples...")
    
    # Test data for checksum calculation
    test_data = b"Sample data for checksum validation in resumable media uploads"
    print(f"Test data: {test_data[:30]}...")
    print(f"Data size: {len(test_data)} bytes")
    
    # Calculate various checksums
    checksums = {}
    
    # MD5 checksum
    md5_hash = hashlib.md5(test_data).hexdigest()
    checksums['md5'] = md5_hash
    print(f"MD5: {md5_hash}")
    
    # CRC32C checksum (if available)
    try:
        import crcmod
        crc32c = crcmod.predefined.mkCrcFun('crc-32c')
        crc_value = crc32c(test_data)
        checksums['crc32c'] = f"{crc_value:08x}"
        print(f"CRC32C: {checksums['crc32c']}")
    except ImportError:
        print("CRC32C: crcmod not available")
    
    # Content range examples
    content_ranges = [
        "bytes 0-255/1024",      # First 256 bytes of 1KB file
        "bytes 256-511/1024",    # Second 256 bytes
        "bytes 512-1023/1024",   # Last 512 bytes
        "bytes */1024"           # Unknown range, known total
    ]
    
    print(f"\nContent-Range header examples:")
    for range_header in content_ranges:
        print(f"  Content-Range: {range_header}")
    
    # Common helper functions (demonstration)
    helper_functions = [
        "_helpers.get_range_info() - Parse range headers",
        "_helpers.get_content_range() - Format range headers",
        "common.TOO_MANY_REQUESTS - HTTP 429 constant",
        "common.UPLOAD_CHUNK_SIZE - Default chunk size"
    ]
    
    print(f"\nHelper utilities:")
    for helper in helper_functions:
        print(f"  • {helper}")
    
    # Upload validation scenarios
    validation_checks = [
        "Content-Length header validation",
        "MD5 checksum verification", 
        "CRC32C integrity checking",
        "Range boundary validation",
        "Total size confirmation"
    ]
    
    print(f"\nValidation checks:")
    for check in validation_checks:
        print(f"  • {check}")
    
    # Error handling constants
    try:
        too_many_requests = common.TOO_MANY_REQUESTS
        print(f"\nHTTP 429 Too Many Requests: {too_many_requests}")
    except AttributeError:
        print("\nHTTP constants not directly available")
    
    print("Example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"google_resumable_media example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: google_resumable_media functionality test 5

In [ ]:
from google.resumable_media import common
from google.resumable_media.requests import ResumableUpload, Download
import io

# Example 5: Error handling and production considerations
print("Example 5: Error handling and production considerations")

try:
    # Production upload workflow demonstration
    print("Setting up production workflow example...")
    
    # Common error scenarios in resumable media
    error_scenarios = {
        "NetworkError": "Connection timeout or network failure",
        "InvalidResponse": "Malformed server response",
        "InvalidChunkSize": "Chunk size not multiple of 256KB",
        "InvalidRange": "Range header parsing error", 
        "TooManyRequests": "Rate limiting (HTTP 429)"
    }
    
    print(f"Common error scenarios:")
    for error, description in error_scenarios.items():
        print(f"  • {error}: {description}")
    
    # Retry strategies
    retry_strategies = [
        "Exponential backoff for rate limits",
        "Resume from last successful chunk",
        "Validate chunk boundaries",
        "Verify upload state before retry",
        "Handle partial upload recovery"
    ]
    
    print(f"\nRetry strategies:")
    for strategy in retry_strategies:
        print(f"  • {strategy}")
    
    # Upload state management
    upload_states = {
        "NOT_STARTED": "Upload not initiated",
        "IN_PROGRESS": "Upload chunks being sent",
        "PAUSED": "Upload temporarily stopped",
        "COMPLETED": "Upload finished successfully",
        "FAILED": "Upload failed, may be recoverable"
    }
    
    print(f"\nUpload state management:")
    for state, description in upload_states.items():
        print(f"  {state}: {description}")
    
    # Performance considerations
    performance_tips = [
        "Use appropriate chunk sizes (256KB-16MB)",
        "Implement proper timeout handling",
        "Monitor upload progress and speed",
        "Handle concurrent upload limits",
        "Validate checksums for data integrity"
    ]
    
    print(f"\nPerformance considerations:")
    for tip in performance_tips:
        print(f"  • {tip}")
    
    # Example production error handling pattern
    print(f"\nProduction error handling pattern:")
    print("  try:")
    print("    response = upload.consume_next_chunk(transport)")
    print("  except requests.exceptions.RequestException as e:")
    print("    if e.response.status_code == 429:")
    print("      time.sleep(exponential_backoff())")
    print("    elif 500 <= e.response.status_code < 600:")
    print("      resume_upload_from_last_chunk()")
    
    # Configuration constants
    try:
        chunk_size = common.UPLOAD_CHUNK_SIZE
        print(f"\nDefault upload chunk size: {chunk_size:,} bytes")
    except AttributeError:
        print(f"\nConstants may vary by version")
    
    print("Example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"google_resumable_media example (cell 10) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")